# EDA — Análise Exploratória dos Ativos Financeiros

> **Propósito:** Este notebook é estritamente exploratório. Nenhum código aqui é trigger de produção.  
> Todo o pipeline produtivo reside em `src/`. O objetivo aqui é entender os dados e justificar as decisões de modelagem.

**Ativos analisados:** PETR4.SA (Petrobras) e NVDC34.SA (NVIDIA BDR)  
**Janela histórica:** 2019 – 2024

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('Ambiente configurado.')

## 1. Carregamento dos Dados

In [ ]:
def load_asset(path: str, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if isinstance(df.columns, pd.MultiIndex) or 'Price' in str(df.columns[0]):
        df = pd.read_csv(path, header=[0, 1])
        df.columns = [c[0] for c in df.columns]
    df.columns = df.columns.str.strip()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    df.name = name
    print(f'{name}: {len(df)} pregões | {df.index.min().date()} → {df.index.max().date()}')
    return df

petr4 = load_asset('../data/raw/petr4_sa_raw.csv', 'PETR4.SA')
nvdc34 = load_asset('../data/raw/nvdc34_sa_raw.csv', 'NVDC34.SA')

## 2. Estatísticas Descritivas

In [ ]:
for name, df in [('PETR4.SA', petr4), ('NVDC34.SA', nvdc34)]:
    print(f'\n=== {name} ===')
    display(df[['Open','High','Low','Close','Volume']].describe().round(2))

## 3. Histórico de Preços de Fechamento

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for ax, (name, df, color) in zip(axes, [
    ('PETR4.SA', petr4, '#2196F3'),
    ('NVDC34.SA', nvdc34, '#FF5722')
]):
    ax.plot(df.index, df['Close'], color=color, linewidth=1.2)
    ax.fill_between(df.index, df['Close'], alpha=0.1, color=color)
    ax.set_title(f'{name} — Preço de Fechamento Histórico')
    ax.set_ylabel('Preço (R$)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../data/processed/eda_preco_historico.png', dpi=120)
plt.show()

## 4. Análise de Retornos Diários e Volatilidade

O retorno logarítmico é a métrica padrão em análise financeira por ser aditivo no tempo e aproximadamente normal.

In [ ]:
petr4['log_return'] = np.log(petr4['Close'] / petr4['Close'].shift(1))
nvdc34['log_return'] = np.log(nvdc34['Close'] / nvdc34['Close'].shift(1))

# Volatilidade rolling de 30 dias (desvio padrão anualizado)
petr4['volatility_30d'] = petr4['log_return'].rolling(30).std() * np.sqrt(252)
nvdc34['volatility_30d'] = nvdc34['log_return'].rolling(30).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

for i, (name, df, color) in enumerate([
    ('PETR4.SA', petr4, '#2196F3'),
    ('NVDC34.SA', nvdc34, '#FF5722')
]):
    axes[i][0].plot(df.index, df['log_return'], color=color, linewidth=0.8, alpha=0.8)
    axes[i][0].set_title(f'{name} — Retorno Log Diário')
    axes[i][0].set_ylabel('Retorno Log')
    axes[i][0].axhline(0, color='black', linewidth=0.5, linestyle='--')

    axes[i][1].plot(df.index, df['volatility_30d'], color=color, linewidth=1.2)
    axes[i][1].set_title(f'{name} — Volatilidade 30d (Anualizada)')
    axes[i][1].set_ylabel('Volatilidade')

plt.tight_layout()
plt.savefig('../data/processed/eda_retornos_volatilidade.png', dpi=120)
plt.show()

print('\nEstatísticas dos Retornos Diários:')
for name, df in [('PETR4.SA', petr4), ('NVDC34.SA', nvdc34)]:
    r = df['log_return'].dropna()
    print(f'  {name}: média={r.mean():.4f} | std={r.std():.4f} | skew={r.skew():.2f} | kurt={r.kurt():.2f}')

## 5. Distribuição dos Preços de Fechamento

Verificamos se os dados seguem distribuição normal — spoiler: não seguem, o que justifica o uso de modelos não-lineares como LSTM em vez de modelos paramétricos clássicos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, df, color) in zip(axes, [
    ('PETR4.SA', petr4, '#2196F3'),
    ('NVDC34.SA', nvdc34, '#FF5722')
]):
    sns.histplot(df['Close'], kde=True, ax=ax, color=color, bins=50)
    ax.set_title(f'{name} — Distribuição do Preço de Fechamento')
    ax.set_xlabel('Preço (R$)')

plt.tight_layout()
plt.savefig('../data/processed/eda_distribuicao.png', dpi=120)
plt.show()

## 6. Autocorrelação (ACF) e Autocorrelação Parcial (PACF)

Esta é a análise mais importante para justificar a **janela de 60 dias** usada na LSTM.  
ACF mede a correlação do preço com seus próprios valores no passado. Se ACF ainda é significativa em lag 60, há memória estatística de ao menos 60 pregões — o que justifica a janela escolhida.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

for i, (name, df) in enumerate([('PETR4.SA', petr4), ('NVDC34.SA', nvdc34)]):
    close = df['Close'].dropna()
    plot_acf(close, lags=80, ax=axes[i][0], title=f'{name} — ACF (Preço de Fechamento)')
    plot_pacf(close, lags=40, ax=axes[i][1], title=f'{name} — PACF')
    axes[i][0].axvline(x=60, color='red', linestyle='--', alpha=0.6, label='lag 60 (nossa janela)')
    axes[i][0].legend()

plt.tight_layout()
plt.savefig('../data/processed/eda_acf_pacf.png', dpi=120)
plt.show()

print('Interpretação: barras além da faixa azul (banda de confiança 95%) indicam autocorrelação estatisticamente significativa.')
print('Se o sinal persiste em lag=60, a janela de 60 dias está bem fundamentada empiricamente.')

## 7. Teste de Estacionariedade (Augmented Dickey-Fuller)

Séries estacionárias são mais fáceis de modelar. Séries de preços brutos geralmente não são estacionárias (têm tendência). O teste ADF confirma isso e justifica o uso de normalização (Min-Max Scaler) antes da LSTM.

In [ ]:
def adf_test(series: pd.Series, name: str):
    result = adfuller(series.dropna(), autolag='AIC')
    p_value = result[1]
    is_stationary = p_value < 0.05
    print(f'{name}:')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {p_value:.6f}')
    print(f'  Estacionária  : {"SIM" if is_stationary else "NÃO — normalização necessária"}')
    print()

adf_test(petr4['Close'], 'PETR4.SA — Preço Bruto')
adf_test(petr4['log_return'], 'PETR4.SA — Retorno Log')
adf_test(nvdc34['Close'], 'NVDC34.SA — Preço Bruto')
adf_test(nvdc34['log_return'], 'NVDC34.SA — Retorno Log')

## 8. Decomposição de Tendência e Sazonalidade

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12))

decomp = seasonal_decompose(petr4['Close'].dropna(), model='multiplicative', period=252)

decomp.observed.plot(ax=axes[0], title='PETR4.SA — Observado', color='#2196F3')
decomp.trend.plot(ax=axes[1], title='Tendência', color='#4CAF50')
decomp.seasonal.plot(ax=axes[2], title='Sazonalidade', color='#FF9800')
decomp.resid.plot(ax=axes[3], title='Resíduo', color='#9E9E9E')

for ax in axes:
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('../data/processed/eda_decomposicao_petr4.png', dpi=120)
plt.show()

## 9. Correlação entre os Ativos

In [ ]:
# Alinhar os dois ativos pelo índice de datas em comum
combined = pd.DataFrame({
    'PETR4_Close': petr4['Close'],
    'NVDC34_Close': nvdc34['Close'],
    'PETR4_Return': petr4['log_return'],
    'NVDC34_Return': nvdc34['log_return'],
}).dropna()

corr = combined.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax)
ax.set_title('Correlação entre PETR4.SA e NVDC34.SA')

plt.tight_layout()
plt.savefig('../data/processed/eda_correlacao.png', dpi=120)
plt.show()

pearson = combined['PETR4_Return'].corr(combined['NVDC34_Return'])
print(f'Correlação de Pearson entre retornos diários: {pearson:.4f}')
print('Valores próximos de 0 indicam que os dois ativos podem se complementar numa carteira diversificada.')

## 10. Volume de Negociação

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

for ax, (name, df, color) in zip(axes, [
    ('PETR4.SA', petr4, '#2196F3'),
    ('NVDC34.SA', nvdc34, '#FF5722')
]):
    vol_rolling = df['Volume'].rolling(30).mean()
    ax.bar(df.index, df['Volume'], color=color, alpha=0.3, label='Volume diário')
    ax.plot(df.index, vol_rolling, color=color, linewidth=1.5, label='Média 30d')
    ax.set_title(f'{name} — Volume de Negociação')
    ax.set_ylabel('Volume')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/eda_volume.png', dpi=120)
plt.show()

## 11. Sumário das Decisões de Modelagem (baseadas nesta EDA)

| Achado | Decisão Tomada |
|---|---|
| Preços brutos **não são estacionários** (ADF p > 0.05) | Aplicar **Min-Max Scaler** antes de alimentar a LSTM |
| ACF significativa além do lag 60 | **Janela de 60 dias** é estatisticamente justificada |
| Distribuição de retornos com **caudas pesadas** (kurtosis > 3) | Modelos lineares são insuficientes — justifica LSTM |
| Volatilidade **não constante** ao longo do tempo | Necessidade de monitoramento de drift em produção |
| Correlação entre ativos **próxima de zero** nos retornos | Modelos independentes por ativo fazem sentido |

> **Nota:** Este notebook foi executado com dados reais de mercado coletados via `yfinance`.  
> Para reproduzir, execute primeiro `poetry run python src/features/data_collection.py --ticker PETR4.SA`.